# Análise Bibliométrica — Teoria das Representações Sociais (2004–2024)

Notebook da pesquisa de doutorado que organiza, trata e analisa o levantamento da produção científica sobre a Teoria das Representações Sociais publicada na revista, no recorte temporal de 2004 a 2024.

**Fonte de dados:** `/Volumes/tese_bibliometria/bronze/landing/Levantamento RS revista Estacio.xlsx` — abas `Artigos`, `Informações completas` e `Geral`.

**Arquitetura:** Bronze (dados de origem preservados) → Silver (limpeza, padronização, validação e integração, gravadas como tabelas Delta no catálogo `tese_bibliometria.silver`) → indicadores bibliométricos (evolução temporal, autoria e palavras-chave).

**Sumário**

1. Leitura dos dados de origem
2. Exploração inicial
3. Camada Bronze
4. Diagnóstico do campo "Ano" (Spark)
5. Tratamento do campo "Ano" (pandas)
6. Silver — Informações completas (recorte 2004–2024)
7. Tratamento e validação da aba Geral
8. Silver integrada — `artigos_bibliometria`
9. Indicadores de autoria
10. Indicadores de palavras-chave

> **Observação de rastreabilidade:** algumas células terminaram em erro durante a exploração original (cast de texto em número, exibição de DataFrame vazio e coluna duplicada no join). Elas foram **mantidas como registro do processo** e estão marcadas como "REGISTRO HISTÓRICO" nos comentários, cada uma com a correção correspondente na sequência. Ver também as observações finais ao fim do notebook.


In [0]:
# LEITURA DOS DADOS DE ORIGEM (BRONZE / LANDING)
# Carrega a planilha do levantamento bibliográfico diretamente do
# Volume de ingressão usando pandas. Três abas são lidas:
#   - "Artigos": dados bibliográficos básicos (título, autor, ano, link);
#   - "Informações completas": dados enriquecidos (UF, instituição,
#     palavras-chave, área de estudo);
#   - "Geral": classificação analítica (tipo, localização, região, tema).
# Os DataFrames pandas são a matéria-prima de todas as etapas seguintes;
# o arquivo de origem não é alterado.
import pandas as pd

file_path = "/Volumes/tese_bibliometria/bronze/landing/Levantamento RS revista Estacio.xlsx"

df_artigos_pd = pd.read_excel(
    file_path,
    sheet_name="Artigos"
)

df_informacoes_pd = pd.read_excel(
    file_path,
    sheet_name="Informações completas"
)

df_geral_pd = pd.read_excel(
    file_path,
    sheet_name="Geral"
)

## 2. Exploração inicial

**Objetivo:** conhecer a estrutura dos dados antes de qualquer transformação — abas existentes no arquivo, primeiras linhas de cada aba e nomes das colunas.

**Resultado esperado:** confirmação dos nomes das abas e das colunas que servem de referência para as seleções da camada Bronze (seção 3).


In [0]:
# Lista as abas existentes no arquivo Excel: confirma que os nomes de
# aba usados na célula anterior ("Artigos", "Informações completas",
# "Geral") estão corretos antes de prosseguir.
import pandas as pd

file_path = "/Volumes/tese_bibliometria/bronze/landing/Levantamento RS revista Estacio.xlsx"

excel = pd.ExcelFile(file_path)

print(excel.sheet_names)

['Quantitativo', 'Artigos', 'Informações completas', 'Regiões organizadas', 'Link', 'Geral', 'Área do conhec_Obj. de pesquisa', 'Regiões 1']


In [0]:
# Amostra inicial (head) das três abas: primeira inspeção visual do
# conteúdo e da forma dos dados, antes de qualquer transformação.
print("ARTIGOS")
display(df_artigos_pd.head())

print("INFORMAÇÕES COMPLETAS")
display(df_informacoes_pd.head())

print("GERAL")
display(df_geral_pd.head())

ARTIGOS


Titulo,Autor,Coautores,Ano,Link,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Edição,Ano.1,Quantidade de artigos
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",Alda Judith Alves-Mazzotti,Tânia Cristina Wilson,v. 1 n. 1 (2004),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/1987,1.0,null,null,null,null,v. 1 n. 1 (2004),2004.0,1.0
Representações sociais: contribuição para um saber sociocultural sem fronteiras,Denise Jodelet,null,v. 1 n. 2 (2004),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2007/1021,3.0,null,null,null,null,v. 1 n. 2 (2004),2004.0,3.0
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,María A. Banchs,null,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2288,null,null,null,null,null,v. 2 n. 3 (2005),2005.0,0.0
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,Mônica Duarte,Tarso Mazzotti,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2290/1100,null,null,null,null,null,v. 2 n. 4 (2005),2005.0,0.0
A pesquisa das representações sociais pode nos enganar,Celso Pereira de Sá,null,v. 5 n. 11 (2008),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7524/47966315,6.0,null,null,null,null,v. 3 n. 5 (2006),2006.0,0.0


INFORMAÇÕES COMPLETAS


Titulo,Autor,Coautores,Ano,Link,UF autor,Instituição UF,Palavras chave,Área de estudo,Unnamed: 9
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",Alda Judith Alves-Mazzotti,Tânia Cristina Wilson,v. 1 n. 1 (2004),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/1987,RJ,UNESA,null,Educação,null
Representações sociais: contribuição para um saber sociocultural sem fronteiras,Denise Jodelet,null,v. 1 n. 2 (2004),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2007/1021,Mexico,UNAM,null,Sociocultural,null
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,María A. Banchs,null,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2288,Venezuela,Universidad Central de Venezuela,null,Estudo da teoria,null
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,Mônica Duarte,Tarso Mazzotti,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2290/1100,RJ,UNIRIO,null,Estudo da teoria,null
A pesquisa das representações sociais pode nos enganar,Celso Pereira de Sá,null,v. 5 n. 11 (2008),https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7524/47966315,RJ,UERJ,null,Estudo da teoria,null


GERAL


Titulo,UF autor,Instituição UF,Tipo de artigo,Instituição (1º autor),Localização,Região,Área de conhecimento do primeiro autor,Temas do objeto de pesquisa,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Instituição UF.1
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",RJ,UNESA,Empírico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Educação / Fracasso escolar,null,null,null,null,UNEB
Representações sociais: contribuição para um saber sociocultural sem fronteiras,Mexico,UNAM,Teórico,UNAM,"Internacional, México",Internacional (México),Sociologia,Cultura / Sociocultural,null,null,null,null,UEPG
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,Venezuela,Universidad Central de Venezuela,Teórico,UFF,"Internacional, Venezuela",Internacional (Venezuela),Psicologia,Teoria / Metodologia,null,null,null,null,UFOPA
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,RJ,UNIRIO,Teórico,UNIRIO,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Teoria / Metodologia,null,null,null,null,Universidade Feevale
A pesquisa das representações sociais pode nos enganar,RJ,UERJ,Teórico,UERJ,"Rio de Janeiro, Brasil",Sudeste (RJ),Psicologia,Epistemologia da TRS,null,null,null,null,UFMG


In [0]:
# Exibe os nomes das colunas de cada aba. Serve de referência para as
# seleções e renomeações feitas na camada Bronze (seção 3).
print("Colunas - ARTIGOS")
print(df_artigos_pd.columns.tolist())

print("\nColunas - INFORMAÇÕES COMPLETAS")
print(df_informacoes_pd.columns.tolist())

print("\nColunas - GERAL")
print(df_geral_pd.columns.tolist())

Colunas - ARTIGOS
['Titulo', 'Autor', 'Coautores', 'Ano', 'Link', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Edição', 'Ano.1', 'Quantidade de artigos']

Colunas - INFORMAÇÕES COMPLETAS
['Titulo', 'Autor', 'Coautores', 'Ano', 'Link', 'UF autor', 'Instituição UF', 'Palavras chave', 'Área de estudo', 'Unnamed: 9']

Colunas - GERAL
['Titulo', 'UF autor', 'Instituição UF', 'Tipo de artigo', 'Instituição (1º autor)', 'Localização', 'Região', 'Área de conhecimento do primeiro autor', 'Temas do objeto de pesquisa', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Instituição UF.1']


## 3. Camada Bronze — seleção e padronização de colunas

**Objetivo:** para cada aba da planilha, selecionar as colunas de interesse da pesquisa e renomeá-las para o padrão snake_case (`titulo`, `autor`, `ano`, `uf_autor`, ...).

**Dados:** DataFrames pandas da seção 1 (dados brutos de origem).

**Resultado esperado:** três DataFrames Bronze (`df_artigos_bronze`, `df_informacoes_bronze`, `df_geral_bronze`) com os **valores originais** da planilha preservados — nenhuma limpeza é feita nesta camada; os tratamentos ocorrem na Silver (seções 4 a 8).


In [0]:
# CAMADA BRONZE — seleção e padronização de colunas (valores originais).
# Para cada aba: seleciona as colunas de interesse da pesquisa e renomeia
# para o padrão snake_case. Nenhuma limpeza de valor é feita aqui — a
# Bronze preserva os dados de origem; o tratamento ocorre na Silver.
# =========================
# BRONZE - ARTIGOS
# =========================

df_artigos_bronze = df_artigos_pd[
    [
        "Titulo",
        "Autor",
        "Coautores",
        "Ano",
        "Link"
    ]
].copy()

df_artigos_bronze.columns = [
    "titulo",
    "autor",
    "coautores",
    "ano",
    "link"
]


# =========================
# BRONZE - INFORMAÇÕES COMPLETAS
# =========================

df_informacoes_bronze = df_informacoes_pd[
    [
        "Titulo",
        "Autor",
        "Coautores",
        "Ano",
        "Link",
        "UF autor",
        "Instituição UF",
        "Palavras chave",
        "Área de estudo"
    ]
].copy()

df_informacoes_bronze.columns = [
    "titulo",
    "autor",
    "coautores",
    "ano",
    "link",
    "uf_autor",
    "instituicao_uf",
    "palavras_chave",
    "area_estudo"
]


# =========================
# BRONZE - GERAL
# =========================

df_geral_bronze = df_geral_pd[
    [
        "Titulo",
        "UF autor",
        "Instituição UF",
        "Tipo de artigo",
        "Instituição (1º autor)",
        "Localização",
        "Região",
        "Área de conhecimento do primeiro autor",
        "Temas do objeto de pesquisa"
    ]
].copy()

df_geral_bronze.columns = [
    "titulo",
    "uf_autor",
    "instituicao_uf",
    "tipo_artigo",
    "instituicao_primeiro_autor",
    "localizacao",
    "regiao",
    "area_conhecimento_primeiro_autor",
    "tema_objeto_pesquisa"
]

## 4. Diagnóstico do campo "Ano" (Spark)

**Objetivo:** investigar por que a coluna `ano` não pode ser usada diretamente: a planilha usa células mescladas, gerando valores mistos — números, textos como `v. 1 n. 1 (2004)` e nulos.

**Procedimento:** criação de visão temporária Spark (`informacoes_completas`); tentativa de recorte `ano <> 2025` (que evidencia o erro de cast); extração do ano por expressão regular; recorte `<= 2024` com `TRY_CAST`.

**Resultado esperado:** identificação do padrão do campo e de uma primeira versão Silver em Spark, posteriormente **substituída** pelo tratamento em pandas da seção 5 (que também corrige as células mescladas com `ffill`). As células com erro nesta seção são registros históricos mantidos.


In [0]:
# Cria a visão temporária Spark "informacoes_completas" a partir da
# Bronze (pandas) e conta os artigos por ano. Nota: o campo "ano" chega
# com valores mistos (números e textos como "v. 1 n. 1 (2004)") por causa
# de células mescladas na planilha — o que motiva o diagnóstico da seção 4.
spark.createDataFrame(df_informacoes_bronze).createOrReplaceTempView("informacoes_completas")

display(spark.sql("""
SELECT
    ano,
    COUNT(*) AS quantidade
FROM informacoes_completas
GROUP BY ano
ORDER BY ano
"""))

ano,quantidade
null,55
v. 1 n. 1 (2004),1
v. 1 n. 2 (2004),1
v. 10 n. 20 (2013),1
v. 10 n. 21 (2013),1
v. 10 n. 22 (2013),1
v. 11 n. 24 (2014),1
v. 11 n. 25 (2014),1
v. 12 n. 27 (2015),1
v. 14 n. 35 (2017),1


In [0]:
# Recria a visão temporária (mesma da célula anterior) para as consultas
# seguintes. Recarga repetida mantida como registro do processo exploratório.
spark.createDataFrame(df_informacoes_bronze).createOrReplaceTempView("informacoes_completas")

In [0]:
# Primeira tentativa de recorte temporal: excluir o ano de 2025.
# O filtro compara "ano" com um número, mas a coluna contém textos
# (ex.: "v. 1 n. 1 (2004)") — a ação desta consulta leva ao erro
# registrado na célula seguinte.
df_informacoes_silver = spark.sql("""
SELECT *
FROM informacoes_completas
WHERE ano <> 2025
   OR ano IS NULL
""")

In [0]:
# REGISTRO HISTÓRICO — esta célula terminou em ERRO na execução original:
# [CAST_INVALID_INPUT] o Spark tentou converter "v. 1 n. 1 (2004)" (texto)
# para número ao agrupar/ordenar. A partir daqui o tratamento do ano passa
# a usar extração por expressão regular (células seguintes) e, depois, o
# preenchimento por ffill em pandas (seção 5).
display(
    df_informacoes_silver.groupBy("ano")
    .count()
    .orderBy("ano")
)

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-6483393535702288>, line 1
----> 1 display(
      2     df_informacoes_silver.groupBy("ano")
      3     .count()
      4     .orderBy("ano")
      5 )

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:115, in Display.display_connect_table(self, df, **kwargs)
    112     self.cf_helper.display_streaming_dataframe(df, config, self.streaming_listener,
    113                                                **kwargs)
    114 else:
--> 115     self.cf_helper.display_dataframe(df, 

In [0]:
# Recria a visão temporária para os testes de extração do ano por regex.
spark.createDataFrame(df_informacoes_bronze).createOrReplaceTempView("informacoes_completas")

In [0]:
# Testa a extração do ano de 4 dígitos (19xx/20xx) da coluna "ano"
# tratando-a como texto (regexp_extract). Evidencia que o ano está sempre
# presente no padrão "v. X n. Y (AAAA)".
display(spark.sql("""
SELECT
    ano,
    regexp_extract(CAST(ano AS STRING), '(19|20)[0-9]{2}', 0) AS ano_extraido
FROM informacoes_completas
"""))

ano,ano_extraido
v. 1 n. 1 (2004),2004
v. 1 n. 2 (2004),2004
null,null
null,null
v. 5 n. 11 (2008),2008
null,null
null,null
null,null
null,null
null,null


In [0]:
# Silver (versão Spark): extrai o ano por regex, converte com TRY_CAST
# (tolera falhas devolvendo NULL) e aplica o recorte temporal da pesquisa:
# apenas artigos até 2024. Esta versão foi substituída, na sequência, pelo
# tratamento em pandas (seção 5), que também corrige os anos ausentes
# devidos a células mescladas (ffill).
df_informacoes_silver = spark.sql("""
SELECT
    titulo,
    autor,
    coautores,

    TRY_CAST(
        regexp_extract(CAST(ano AS STRING), '(19|20)[0-9]{2}', 0)
        AS INT
    ) AS ano,

    link,
    uf_autor,
    instituicao_uf,
    palavras_chave,
    area_estudo

FROM informacoes_completas

WHERE TRY_CAST(
        regexp_extract(CAST(ano AS STRING), '(19|20)[0-9]{2}', 0)
        AS INT
      ) <= 2024
""")

In [0]:
# Confere a distribuição de artigos por ano da Silver em Spark
# (versão sem a correção de células mescladas).
display(
    df_informacoes_silver
    .groupBy("ano")
    .count()
    .orderBy("ano")
)

ano,count
2004,2
2008,1
2010,2
2011,1
2013,3
2014,2
2015,1
2017,4
2018,1
2019,2


In [0]:
# Lista os registros com "ano" nulo na visão: evidencia que a planilha
# usa células mescladas (o ano aparece apenas na primeira linha do grupo),
# o que justifica o preenchimento com ffill na seção 5.
display(
    spark.sql("""
    SELECT
        titulo,
        autor,
        ano,
        link
    FROM informacoes_completas
    WHERE ano IS NULL
    """)
)

titulo,autor,ano,link
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,María A. Banchs,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2288
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,Mônica Duarte,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2290/1100
Reflexiones acerca de las prácticas docentes desde la perspectiva de la Teorí­a de las Representaciones Sociales,Susana Seidmann,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7277/47966194
Contribuições da Psicologia Sócio-Histórica ao estudo da teoria das Representações Sociais,Wanda Maria Junqueira de Aguiar,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7428/47966310
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,Claudia Rabello de Castro,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7521/47966311
Para uma “pedagogia das representações sociais”,Tarso Mazzotti,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7522/47966312
Perspectivas teórico-metodológicas em representações sociais,Rita de Cássia Pereira Lima,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7523/47966313
A hipotese das Representações Sociais hegemónicas do trabalho docente,Pedro Humberto Faria Campos,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/777/402
Representações Sociais do trabalho docente: um olhar sobre a subjetividade do professor em sala de aula,Monica Rabello de Castro,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/765/406
"Família atual, futura e ideal: zeitgeist e Representações sociais",Annamaria Silvana de Rosa,null,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/936/486


## 5. Tratamento do campo "Ano" (pandas)

**Objetivo:** produzir uma coluna `ano` numérica e confiável a partir do texto original da planilha: extrair o ano de 4 dígitos (19xx/20xx) e preencher os valores ausentes — decorrentes de células mescladas — com o último ano válido anterior (`ffill`).

**Dados:** `df_informacoes_bronze` (cópia; o DataFrame de origem não é alterado).

**Resultado esperado:** `df_informacoes_tratado` com a coluna `ano_extraido` numérica (tipos Int64), validada visualmente lado a lado com o valor original. O recorte temporal é aplicado na seção 6.


In [0]:
# TRATAMENTO DO CAMPO "ANO" (pandas)
# Copia a Bronze (sem alterar os dados de origem) e cria a coluna
# "ano_extraido": extrai o ano de 4 dígitos (19xx/20xx) do texto original
# e preenche os valores ausentes com o último ano válido anterior (ffill),
# corrigindo as células mescladas da planilha. Converte para inteiro (Int64,
# que aceita nulos caso algum registro não tenha ano extraível).
import pandas as pd

# Faz uma cópia para não alterar o dataframe original
df_informacoes_tratado = df_informacoes_bronze.copy()

# Extrai o ano de 4 dígitos
df_informacoes_tratado["ano_extraido"] = (
    df_informacoes_tratado["ano"]
    .astype("string")
    .str.extract(r"((?:19|20)\d{2})")[0]
)

# Preenche os anos ausentes com o último ano válido acima
df_informacoes_tratado["ano_extraido"] = (
    df_informacoes_tratado["ano_extraido"]
    .ffill()
)

# Converte para número
df_informacoes_tratado["ano_extraido"] = pd.to_numeric(
    df_informacoes_tratado["ano_extraido"],
    errors="coerce"
).astype("Int64")

In [0]:
# Inspeciona o resultado do tratamento: título, ano original e ano
# extraído lado a lado, para validar visualmente a correção por ffill.
display(
    df_informacoes_tratado[
        ["titulo", "ano", "ano_extraido"]
    ]
)

titulo,ano,ano_extraido
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",v. 1 n. 1 (2004),2004
Representações sociais: contribuição para um saber sociocultural sem fronteiras,v. 1 n. 2 (2004),2004
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,null,2004
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,null,2004
A pesquisa das representações sociais pode nos enganar,v. 5 n. 11 (2008),2008
Reflexiones acerca de las prácticas docentes desde la perspectiva de la Teorí­a de las Representaciones Sociales,null,2008
Contribuições da Psicologia Sócio-Histórica ao estudo da teoria das Representações Sociais,null,2008
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,null,2008
Para uma “pedagogia das representações sociais”,null,2008
Perspectivas teórico-metodológicas em representações sociais,null,2008


In [0]:
# Célula DUPLICADA da anterior (mesmo tratamento do ano: extração por
# regex + ffill + conversão para Int64). Mantida como registro do processo;
# o resultado é idêntico ao já obtido — ver redundância nas observações finais.
df_informacoes_tratado = df_informacoes_bronze.copy()

# Extrai o ano
df_informacoes_tratado["ano_extraido"] = (
    df_informacoes_tratado["ano"]
    .astype("string")
    .str.extract(r"((?:19|20)\d{2})")[0]
)

# Preenche anos mesclados
df_informacoes_tratado["ano_extraido"] = (
    df_informacoes_tratado["ano_extraido"]
    .ffill()
)

# Converte ano para número
df_informacoes_tratado["ano_extraido"] = pd.to_numeric(
    df_informacoes_tratado["ano_extraido"],
    errors="coerce"
).astype("Int64")

### 5.1 Tratamento paralelo da aba "Artigos" (registro exploratório)

**Objetivo:** aplicar à aba "Artigos" o mesmo tratamento do campo ano (extração de 4 dígitos + `ffill`), incluindo também a coluna "Edição".

**Resultado esperado:** `df_artigos_tratado` com `ano` numérico e edição preenchida. Este DataFrame **não alimenta as tabelas Silver gravadas neste notebook** — é mantido como registro do levantamento.


In [0]:
# Tratamento paralelo da aba "Artigos" (registro exploratório, não usado
# nas tabelas Silver finais): seleciona as colunas de interesse — incluindo
# "Edição", exclusiva desta aba — e padroniza os nomes para snake_case.
df_artigos_tratado = df_artigos_pd[
    [
        "Titulo",
        "Autor",
        "Coautores",
        "Ano",
        "Edição",
        "Link"
    ]
].copy()

df_artigos_tratado.columns = [
    "titulo",
    "autor",
    "coautores",
    "ano_original",
    "edicao",
    "link"
]

In [0]:
# Preenche para baixo (ffill) "ano_original" e "edicao": a aba Artigos
# também usa células mescladas por edição na planilha de origem.
df_artigos_tratado["ano_original"] = (
    df_artigos_tratado["ano_original"]
    .ffill()
)

df_artigos_tratado["edicao"] = (
    df_artigos_tratado["edicao"]
    .ffill()
)

In [0]:
# Extrai o ano de 4 dígitos de "ano_original" e converte para inteiro,
# garantindo uma coluna "ano" analítica na aba Artigos.
df_artigos_tratado["ano"] = (
    df_artigos_tratado["ano_original"]
    .astype("string")
    .str.extract(r"((?:19|20)\d{2})")[0]
)

df_artigos_tratado["ano"] = pd.to_numeric(
    df_artigos_tratado["ano"],
    errors="coerce"
).astype("Int64")

In [0]:
# Amostra do resultado do tratamento da aba Artigos (título, ano
# original, ano extraído e edição preenchida).
display(
    df_artigos_tratado[
        ["titulo", "ano_original", "ano", "edicao"]
    ]
)

titulo,ano_original,ano,edicao
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",v. 1 n. 1 (2004),2004,v. 1 n. 1 (2004)
Representações sociais: contribuição para um saber sociocultural sem fronteiras,v. 1 n. 2 (2004),2004,v. 1 n. 2 (2004)
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,v. 1 n. 2 (2004),2004,v. 2 n. 3 (2005)
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,v. 1 n. 2 (2004),2004,v. 2 n. 4 (2005)
A pesquisa das representações sociais pode nos enganar,v. 5 n. 11 (2008),2008,v. 3 n. 5 (2006)
Reflexiones acerca de las prácticas docentes desde la perspectiva de la Teorí­a de las Representaciones Sociales,v. 5 n. 11 (2008),2008,v. 3 n. 6 (2006)
Contribuições da Psicologia Sócio-Histórica ao estudo da teoria das Representações Sociais,v. 5 n. 11 (2008),2008,v. 4 n. 7 (2007)
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,v. 5 n. 11 (2008),2008,v. 4 n. 8 (2007)
Para uma “pedagogia das representações sociais”,v. 5 n. 11 (2008),2008,v. 5 n. 9 (2008)
Perspectivas teórico-metodológicas em representações sociais,v. 5 n. 11 (2008),2008,v. 5 n. 10 (2008)


## 6. Silver — Informações completas (recorte 2004–2024)

**Objetivo:** consolidar a versão tratada da aba "Informações completas", aplicar o critério de inclusão temporal da pesquisa (artigos com ano até 2024) e preparar a gravação como tabela Delta no catálogo.

**Dados:** `df_informacoes_tratado` (seção 5).

**Procedimento:** filtro `ano_extraido <= 2024`; exclusão da coluna `ano` original (texto com células mescladas) e promoção de `ano_extraido` a `ano`.

**Resultado esperado:** `df_informacoes_silver_pd` com 89 artigos distribuídos por ano de 2004 a 2024 (nenhum artigo de 2025), verificado por contagens antes da gravação.


In [0]:
# Distribuição de artigos por ano após a extração + ffill (ainda sem o
# recorte temporal, aplicado na célula seguinte).
display(
    df_informacoes_tratado
    .groupby("ano_extraido")
    .size()
    .reset_index(name="quantidade")
)

ano_extraido,quantidade
2004,4
2008,6
2010,2
2011,1
2013,5
2014,8
2015,2
2017,17
2018,4
2019,3


In [0]:
# Aplica o critério de inclusão da pesquisa: mantém apenas os artigos
# com ano extraído <= 2024 (recorte temporal 2004–2024).
df_informacoes_silver_pd = df_informacoes_tratado[
    df_informacoes_tratado["ano_extraido"] <= 2024
].copy()

In [0]:
# Finaliza a Silver em pandas: remove a coluna "ano" original (texto,
# com células mescladas) e promove "ano_extraido" a "ano".
df_informacoes_silver_pd = df_informacoes_silver_pd.drop(columns=["ano"])

df_informacoes_silver_pd = df_informacoes_silver_pd.rename(
    columns={"ano_extraido": "ano"}
)

In [0]:
# Distribuição por ano da Silver após o recorte e a renomeação.
display(
    df_informacoes_silver_pd
    .groupby("ano")
    .size()
    .reset_index(name="quantidade")
)

ano,quantidade
2004,4
2008,6
2010,2
2011,1
2013,5
2014,8
2015,2
2017,17
2018,4
2019,3


In [0]:
# REGISTRO HISTÓRICO — esta célula terminou em ERRO na execução original:
# [CANNOT_INFER_EMPTY_SCHEMA] ao exibir um DataFrame vazio (não há artigos
# de 2025 — todos foram excluídos pelo recorte temporal). A verificação
# equivalente, sem erro, está na célula seguinte (contagem por len()).
display(
    df_informacoes_silver_pd[
        df_informacoes_silver_pd["ano"] == 2025
    ]
)

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-6483393535702320>, line 1
----> 1 display(
      2     df_informacoes_silver_pd[
      3         df_informacoes_silver_pd["ano"] == 2025
      4     ]
      5 )

File /databricks/python_shell/lib/dbruntime/display.py:208, in Display.display(self, input, *args, **kwargs)
    206     self.display(input.make_dataframe())
    207 elif type(input).__module__ == 'pandas.core.frame' and type(input).__name__ == 'DataFrame':
--> 208     self._display_pandas_dataframe(input)
    209 elif type(input).__module__ in ('databricks.koalas.frame', 'pyspark.pandas.frame') and \
    210         type(input).__name__ == 'DataFrame':
    211     index_col = kwargs.get('index_col')

File /databricks/python_shell/lib/dbruntime/display.py:137, in Display._display_pandas_dataframe(self, input)
    135 def _display_pandas_dataframe(self, input) -> N

In [0]:
# Verificação alternativa (sem display): confirma que não restou nenhum
# artigo de 2025 na Silver após o recorte temporal.
quantidade_2025 = len(
    df_informacoes_silver_pd[
        df_informacoes_silver_pd["ano"] == 2025
    ]
)

print("Quantidade de artigos de 2025:", quantidade_2025)

Quantidade de artigos de 2025: 0


In [0]:
# Total de artigos da Silver: 89 registros no recorte 2004–2024.
print("Total de artigos após excluir 2025:", len(df_informacoes_silver_pd))

Total de artigos após excluir 2025: 89


In [0]:
# Distribuição definitiva por ano (mesmo resultado da célula anterior,
# exibido como conferência final antes de gravar a tabela Silver).
display(
    df_informacoes_silver_pd
    .groupby("ano")
    .size()
    .reset_index(name="quantidade")
)

ano,quantidade
2004,4
2008,6
2010,2
2011,1
2013,5
2014,8
2015,2
2017,17
2018,4
2019,3


In [0]:
# Converte a Silver de pandas para Spark, preparando a gravação
# como tabela Delta gerenciada no catálogo.
df_informacoes_silver = spark.createDataFrame(
    df_informacoes_silver_pd
)

### 6.1 Persistência e validação da Silver

**Objetivo:** converter a Silver de pandas para Spark, gravar a tabela Delta `tese_bibliometria.silver.informacoes_completas` (modo overwrite) e validar os totais por ano diretamente na tabela gravada.

**Resultado esperado:** 89 artigos na tabela, com a mesma distribuição por ano obtida antes da gravação (confirmação de que a gravação preservou os dados).


In [0]:
# Grava a Silver como tabela Delta "tese_bibliometria.silver.informacoes_completas"
# (modo overwrite: substitui a versão anterior). Fonte oficial das
# análises das seções seguintes.
df_informacoes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.informacoes_completas"
    )

In [0]:
# Validação pós-gravação: consulta SQL direto na tabela Delta para
# confirmar os mesmos totais por ano obtidos antes de gravar.
display(
    spark.sql("""
        SELECT ano, COUNT(*) AS quantidade
        FROM tese_bibliometria.silver.informacoes_completas
        GROUP BY ano
        ORDER BY ano
    """)
)

ano,quantidade
2004,4
2008,6
2010,2
2011,1
2013,5
2014,8
2015,2
2017,17
2018,4
2019,3


## 7. Tratamento e validação da aba "Geral"

**Objetivo:** padronizar a aba Geral (classificação analítica: tipo de artigo, instituição do primeiro autor, localização, região, área de conhecimento e tema do objeto de pesquisa) e validar sua qualidade — valores nulos, duplicidades de título e conciliação de totais com a Silver.

**Dados:** `df_geral_pd` (aba Geral da planilha de origem).

**Resultado esperado:** `df_geral_tratado` com 90 artigos (um a mais que a Silver: o artigo de 2025, excluído pelo recorte temporal), sem títulos duplicados, com ausências identificadas em `uf_autor` (1 caso) e `tema_objeto_pesquisa` (50 casos) — registradas, mas não corrigidas nesta revisão.


In [0]:
# TRATAMENTO DA ABA "GERAL"
# Repete a seleção/padronização já feita na camada Bronze (bloco GERAL da
# seção 3) — redundância mantida como registro do processo. O DataFrame
# resultante (90 linhas) classifica cada artigo por tipo, localização,
# região, área de conhecimento e tema do objeto de pesquisa.
df_geral_tratado = df_geral_pd[
    [
        "Titulo",
        "UF autor",
        "Instituição UF",
        "Tipo de artigo",
        "Instituição (1º autor)",
        "Localização",
        "Região",
        "Área de conhecimento do primeiro autor",
        "Temas do objeto de pesquisa"
    ]
].copy()

df_geral_tratado.columns = [
    "titulo",
    "uf_autor",
    "instituicao_uf",
    "tipo_artigo",
    "instituicao_primeiro_autor",
    "localizacao",
    "regiao",
    "area_conhecimento_primeiro_autor",
    "tema_objeto_pesquisa"
]

In [0]:
# Amostra das 20 primeiras linhas classificadas (inspeção visual).
display(df_geral_tratado.head(20))

titulo,uf_autor,instituicao_uf,tipo_artigo,instituicao_primeiro_autor,localizacao,regiao,area_conhecimento_primeiro_autor,tema_objeto_pesquisa
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",RJ,UNESA,Empírico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Educação / Fracasso escolar
Representações sociais: contribuição para um saber sociocultural sem fronteiras,Mexico,UNAM,Teórico,UNAM,"Internacional, México",Internacional (México),Sociologia,Cultura / Sociocultural
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,Venezuela,Universidad Central de Venezuela,Teórico,UFF,"Internacional, Venezuela",Internacional (Venezuela),Psicologia,Teoria / Metodologia
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,RJ,UNIRIO,Teórico,UNIRIO,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Teoria / Metodologia
A pesquisa das representações sociais pode nos enganar,RJ,UERJ,Teórico,UERJ,"Rio de Janeiro, Brasil",Sudeste (RJ),Psicologia,Epistemologia da TRS
Reflexiones acerca de las prácticas docentes desde la perspectiva de la Teorí­a de las Representaciones Sociales,Argentina,UBA,Empírico,UBA,"Internacional, Argentina",Internacional (Argentina),Educação,Práticas docentes
Contribuições da Psicologia Sócio-Histórica ao estudo da teoria das Representações Sociais,SP,PUCSP,Teórico,USP,"São Paulo, Brasil",Sudeste (SP),Psicologia,Psicologia Sócio-Histórica / TRS
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,null,FAP,Empírico,UFBA,"Bahia, Brasil",Nordeste (BA),Serviço Social,ECA / Infância / Assistência social
Para uma “pedagogia das representações sociais”,RJ,UNESA,Teórico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Pedagogia / TRS
Perspectivas teórico-metodológicas em representações sociais,RJ,UNESA,Teórico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Psicologia,Teoria / Metodologia


In [0]:
# Diagnóstico de valores nulos por coluna: "uf_autor" tem 1 valor
# ausente e "tema_objeto_pesquisa" tem 50 — analisados nas células seguintes.
print(df_geral_tratado.isnull().sum())

titulo                               0
uf_autor                             1
instituicao_uf                       0
tipo_artigo                          0
instituicao_primeiro_autor           0
localizacao                          0
regiao                               0
area_conhecimento_primeiro_autor     0
tema_objeto_pesquisa                50
dtype: int64


In [0]:
# Registro com "uf_autor" nulo: artigo da UFBA em que a UF não foi
# preenchida na planilha (o restante do cadastro está completo).
display(
    df_geral_tratado[
        df_geral_tratado["uf_autor"].isna()
    ]
)

titulo,uf_autor,instituicao_uf,tipo_artigo,instituicao_primeiro_autor,localizacao,regiao,area_conhecimento_primeiro_autor,tema_objeto_pesquisa
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,null,FAP,Empírico,UFBA,"Bahia, Brasil",Nordeste (BA),Serviço Social,ECA / Infância / Assistência social


In [0]:
# Artigos sem "tema do objeto de pesquisa" (50 casos): verificação dos
# demais campos para avaliar se a ausência compromete a análise.
# Nenhum preenchimento foi realizado nesta revisão (decisão da pesquisa).
display(
    df_geral_tratado[
        df_geral_tratado["tema_objeto_pesquisa"].isna()
    ][
        [
            "titulo",
            "tipo_artigo",
            "area_conhecimento_primeiro_autor",
            "tema_objeto_pesquisa"
        ]
    ]
)

titulo,tipo_artigo,area_conhecimento_primeiro_autor,tema_objeto_pesquisa
Os jovens da EJA e o trabalho: repensando a prática a partir das expectativas e representações sociais,Empírico,Educação,null
Representações sociais em movimento: uma análise de duas pesquisas no âmbito da Educação do Campo da FaE-UFMG,Empírico,Educação,null
"Abordagem estrutural das representações sociais: da análise de similitude ao grupo focal, uma proposta metodológica",Teórico,Teoria,null
"""Kit Escola Sem Homofobia"" e Discursos Polí­ticos: quando representações sociais se tornam ideológicas",Empírico,Educação,null
Representação social do desenvolvimento psicomotor por professor de Educação Fí­sica Escolar,Empírico,Educação,null
A teoria da representação social e o discurso do sujeito coletivo em estudos no campo da polí­tica educacional: sentidos da interdisciplinaridade nos BI.,Empírico,Educação,null
A Teoria das Representações Sociais: conversando com os professores sobre trabalho,Empírico,Educação,null
Representações sociais: zona muda. Contributos do Modelo de Estratégia Argumentativa,Teórico,Teoria,null
Famí­lias de estudantes de escola pública: um estudo sobre relações entre representações sociais e práticas de professoras,Empírico,Educação,null
Representação social de pedagogas sobre a indisciplina escolar,Empírico,Educação,null


In [0]:
# REGISTRO HISTÓRICO — esta célula terminou em ERRO na execução original:
# [CANNOT_INFER_EMPTY_SCHEMA] ao exibir um DataFrame vazio (não há títulos
# duplicados). A verificação sem erro está na célula seguinte (contagem).
duplicados = df_geral_tratado[
    df_geral_tratado.duplicated(subset=["titulo"], keep=False)
]

display(duplicados)

---------------------------------------------------------------------------
PySparkValueError                         Traceback (most recent call last)
File <command-6483393535702342>, line 5
      1 duplicados = df_geral_tratado[
      2     df_geral_tratado.duplicated(subset=["titulo"], keep=False)
      3 ]
----> 5 display(duplicados)

File /databricks/python_shell/lib/dbruntime/display.py:208, in Display.display(self, input, *args, **kwargs)
    206     self.display(input.make_dataframe())
    207 elif type(input).__module__ == 'pandas.core.frame' and type(input).__name__ == 'DataFrame':
--> 208     self._display_pandas_dataframe(input)
    209 elif type(input).__module__ in ('databricks.koalas.frame', 'pyspark.pandas.frame') and \
    210         type(input).__name__ == 'DataFrame':
    211     index_col = kwargs.get('index_col')

File /databricks/python_shell/lib/dbruntime/display.py:137, in Display._display_pandas_dataframe(self, input)
    135 def _display_pandas_dataframe(self

In [0]:
# Confirma por contagem: nenhum título duplicado na aba Geral
# (as 90 linhas correspondem a 90 artigos distintos).
duplicados = df_geral_tratado[
    df_geral_tratado.duplicated(subset=["titulo"], keep=False)
]

print("Quantidade de títulos duplicados:", len(duplicados))

Quantidade de títulos duplicados: 0


In [0]:
# Conciliação entre as fontes: a aba Geral tem 90 artigos e a Silver das
# Informações completas tem 89 — a diferença é o artigo de 2025, excluído
# pelo recorte temporal da pesquisa.
print("Total Geral:", len(df_geral_tratado))
print("Total Informações Silver:", len(df_informacoes_silver_pd))

Total Geral: 90
Total Informações Silver: 89


In [0]:
# Converte a aba Geral tratada para Spark, preparando o cruzamento
# com a Silver das Informações completas (seção 8).
df_geral_spark = spark.createDataFrame(df_geral_tratado)

## 8. Silver integrada — `artigos_bibliometria`

**Objetivo:** cruzar a Silver das Informações completas (89 artigos, recorte 2004–2024) com a aba Geral por `titulo` (left join), anexando a classificação analítica (tipo de artigo, localização, região, área de conhecimento, tema) a cada artigo, e gravar o resultado como tabela Delta.

**Dados:** `df_informacoes_silver` (seção 6) e `df_geral_spark` (seção 7).

**Procedimento:** o primeiro join gera colunas homônimas (`uf_autor`, `instituicao_uf`) nas duas bases, o que impede a gravação (erro histórico mantido como registro); o join é refeito selecionando explicitamente as colunas de cada lado (aliases `i` e `g`).

**Resultado esperado:** `tese_bibliometria.silver.artigos_bibliometria` com 89 artigos e 15 colunas, todos com correspondência na aba Geral.


In [0]:
# SILVER INTEGRADA — primeiro cruzamento: left join da Silver das
# Informações completas com a aba Geral por "titulo", anexando a
# classificação analítica a cada artigo. Nota: como ambas as bases têm
# "uf_autor" e "instituicao_uf", o resultado fica com colunas duplicadas
# (o que levará ao erro de gravação registrado adiante).
df_bibliometria = (
    df_informacoes_silver
    .alias("i")
    .join(
        df_geral_spark.alias("g"),
        on="titulo",
        how="left"
    )
)

In [0]:
# Verificação do join: 89 linhas — o left join preserva todas as
# linhas da base da esquerda (Silver, 89 artigos).
print("Total após o join:", df_bibliometria.count())

Total após o join: 89


In [0]:
# Procura artigos sem correspondência na aba Geral ("tipo_artigo"
# nulo após o left join) — resultado vazio.
display(
    df_bibliometria.filter(
        "tipo_artigo IS NULL"
    )
)

titulo,autor,coautores,link,uf_autor,instituicao_uf,palavras_chave,area_estudo,ano,uf_autor,instituicao_uf,tipo_artigo,instituicao_primeiro_autor,localizacao,regiao,area_conhecimento_primeiro_autor,tema_objeto_pesquisa


In [0]:
# Confirma por contagem: todos os 89 artigos da Silver têm
# correspondência na aba Geral.
sem_correspondencia = df_bibliometria.filter(
    "tipo_artigo IS NULL"
).count()

print("Artigos sem correspondência no Geral:", sem_correspondencia)

Artigos sem correspondência no Geral: 0


In [0]:
# REGISTRO HISTÓRICO — esta célula terminou em ERRO na execução original:
# [COLUMN_ALREADY_EXISTS] as colunas "instituicao_uf" e "uf_autor" existem
# nas duas bases do join, o que impede a gravação. A gravação só foi
# possível após a célula seguinte, que refaz o join selecionando as
# colunas de cada lado sem ambiguidade.
df_bibliometria.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.artigos_bibliometria"
    )

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6483393535702351>, line 4
      1 df_bibliometria.write \
      2     .format("delta") \
      3     .mode("overwrite") \
----> 4     .saveAsTable(
      5         "tese_bibliometria.silver.artigos_bibliometria"
      6     )

File <command-6483393535702347>, line 2
      1 df_bibliometria = (
----> 2     df_informacoes_silver
      3     .alias("i")
      4     .join(
      5         df_geral_spark.alias("g"),
      6         on="titulo",
      7         how="left"
      8     )
      9 )

File <command-6483393535702324>, line 1
----> 1 df_informacoes_silver = spark.createDataFrame(
      2     df_informacoes_silver_pd
      3 )

File <command-6483393535702346>, line 1
----> 1 df_geral_spark = spark.createDataFrame(df_geral_tratado)

AnalysisException: [COLUMN_ALREADY_EXISTS] The column `instituicao_uf` already exists. Ch

In [0]:
# Refaz o join selecionando explicitamente as colunas de cada lado
# (aliases "i" = Informações completas, "g" = Geral), resolvendo a
# duplicidade de "uf_autor" e "instituicao_uf" que impediu a gravação.
# A coluna "titulo" é a chave do join e vem sem alias.
from pyspark.sql import functions as F

df_bibliometria = (
    df_informacoes_silver.alias("i")
    .join(
        df_geral_spark.alias("g"),
        on="titulo",
        how="left"
    )
    .select(
        F.col("titulo"),

        # Informações completas
        F.col("i.autor").alias("autor"),
        F.col("i.coautores").alias("coautores"),
        F.col("i.ano").alias("ano"),
        F.col("i.link").alias("link"),
        F.col("i.uf_autor").alias("uf_autor"),
        F.col("i.instituicao_uf").alias("instituicao_uf"),
        F.col("i.palavras_chave").alias("palavras_chave"),
        F.col("i.area_estudo").alias("area_estudo"),

        # Aba Geral
        F.col("g.tipo_artigo").alias("tipo_artigo"),
        F.col("g.instituicao_primeiro_autor").alias("instituicao_primeiro_autor"),
        F.col("g.localizacao").alias("localizacao"),
        F.col("g.regiao").alias("regiao"),
        F.col("g.area_conhecimento_primeiro_autor").alias("area_conhecimento_primeiro_autor"),
        F.col("g.tema_objeto_pesquisa").alias("tema_objeto_pesquisa")
    )
)

In [0]:
# Confere as 15 colunas finais da Silver integrada, sem duplicidades.
print(df_bibliometria.columns)

['titulo', 'autor', 'coautores', 'ano', 'link', 'uf_autor', 'instituicao_uf', 'palavras_chave', 'area_estudo', 'tipo_artigo', 'instituicao_primeiro_autor', 'localizacao', 'regiao', 'area_conhecimento_primeiro_autor', 'tema_objeto_pesquisa']


In [0]:
# Total da Silver integrada: 89 artigos.
print("Total de artigos:", df_bibliometria.count())

Total de artigos: 89


In [0]:
# Grava a Silver integrada como tabela Delta
# "tese_bibliometria.silver.artigos_bibliometria" (modo overwrite).
df_bibliometria.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.artigos_bibliometria"
    )

In [0]:
# Validação pós-gravação: conta os registros direto na tabela Delta.
print(
    spark.table(
        "tese_bibliometria.silver.artigos_bibliometria"
    ).count()
)

89


In [0]:
# Amostra ordenada por ano e título: conferência final do conteúdo
# gravado na Silver integrada (20 primeiros registros).
display(
    spark.sql("""
        SELECT *
        FROM tese_bibliometria.silver.artigos_bibliometria
        ORDER BY ano, titulo
        LIMIT 20
    """)
)

titulo,autor,coautores,ano,link,uf_autor,instituicao_uf,palavras_chave,area_estudo,tipo_artigo,instituicao_primeiro_autor,localizacao,regiao,area_conhecimento_primeiro_autor,tema_objeto_pesquisa
Alternativas de apropriação teórica: abordagem processual e estrutural das representações sociais,María A. Banchs,null,2004,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2288,Venezuela,Universidad Central de Venezuela,null,Estudo da teoria,Teórico,UFF,"Internacional, Venezuela",Internacional (Venezuela),Psicologia,Teoria / Metodologia
Análise retórica do discurso como proposta metodológicapara as pesquisas em representação social,Mônica Duarte,Tarso Mazzotti,2004,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2290/1100,RJ,UNIRIO,null,Estudo da teoria,Teórico,UNIRIO,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Teoria / Metodologia
"Relação entre representações sociais de ""fracasso escolar"" de professores do ensino fundamental e sua prática docente",Alda Judith Alves-Mazzotti,Tânia Cristina Wilson,2004,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/1987,RJ,UNESA,null,Educação,Empírico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Educação / Fracasso escolar
Representações sociais: contribuição para um saber sociocultural sem fronteiras,Denise Jodelet,null,2004,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/2007/1021,Mexico,UNAM,null,Sociocultural,Teórico,UNAM,"Internacional, México",Internacional (México),Sociologia,Cultura / Sociocultural
A pesquisa das representações sociais pode nos enganar,Celso Pereira de Sá,null,2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7524/47966315,RJ,UERJ,null,Estudo da teoria,Teórico,UERJ,"Rio de Janeiro, Brasil",Sudeste (RJ),Psicologia,Epistemologia da TRS
As representações sociais de profissionais da assistência sobre o Estatuto da Criança e do Adolescente: estratégias e argumentos,Claudia Rabello de Castro,Monica Rabello de Castro,2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7521/47966311,null,FAP,null,null,Empírico,UFBA,"Bahia, Brasil",Nordeste (BA),Serviço Social,ECA / Infância / Assistência social
Contribuições da Psicologia Sócio-Histórica ao estudo da teoria das Representações Sociais,Wanda Maria Junqueira de Aguiar,Ana Mercês Bahia Bock,2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7428/47966310,SP,PUCSP,null,Estudo da teoria,Teórico,USP,"São Paulo, Brasil",Sudeste (SP),Psicologia,Psicologia Sócio-Histórica / TRS
Para uma “pedagogia das representações sociais”,Tarso Mazzotti,null,2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7522/47966312,RJ,UNESA,null,null,Teórico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Educação,Pedagogia / TRS
Perspectivas teórico-metodológicas em representações sociais,Rita de Cássia Pereira Lima,null,2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7523/47966313,RJ,UNESA,null,null,Teórico,UNESA,"Rio de Janeiro, Brasil",Sudeste (RJ),Psicologia,Teoria / Metodologia
Reflexiones acerca de las prácticas docentes desde la perspectiva de la Teorí­a de las Representaciones Sociales,Susana Seidmann,"Sandra Thomé, Jorgelina Di Iorio, Suzana Azzollini",2008,https://mestradoedoutoradoestacio.periodicoscientificos.com.br/index.php/reeduc/article/view/7277/47966194,Argentina,UBA,null,Educação,Empírico,UBA,"Internacional, Argentina",Internacional (Argentina),Educação,Práticas docentes


## 9. Indicadores de autoria

**Objetivo:** construir a base artigo × autor (autor principal + coautores separados por vÍrgula), padronizar grafias do mesmo autor e produzir a contagem de participações por autor.

**Dados:** tabela Silver `tese_bibliometria.silver.artigos_bibliometria_tratada`.

> **Ponto de atenção (rastreabilidade):** esta tabela **não é criada neste notebook** — na seção 8 grava-se `artigos_bibliometria`, sem o sufixo `_tratada`. A origem da versão `_tratada` precisa ser documentada/confirmada (ponto de validação manual).

**Resultado esperado:** base com 208 registros de autoria (artigo × autor), 89 artigos e 170 autores distintos após padronização de grafia, gravada como `tese_bibliometria.silver.autores_artigos` (sem os registros placeholder "-").


In [0]:
# INDICADORES DE AUTORIA — construção da base artigo × autor.
# Base: tabela Silver "tese_bibliometria.silver.artigos_bibliometria_tratada"
# (ATENÇÃO: tabela não criada neste notebook — dependência externa a
# documentar; ver observações finais).
# 1) Autor principal: uma linha por artigo (coluna "autor").
# 2) Coautores: apenas artigos com coautores preenchidos; a coluna
#    "coautores" (texto separado por vírgula) é dividida e explodida,
#    gerando uma linha por coautor.
# 3) União dos dois conjuntos em "df_todos_autores", cada linha com
#    "tipo_autoria" = principal ou coautor.
from pyspark.sql import functions as F

df_base = spark.table(
    "tese_bibliometria.silver.artigos_bibliometria_tratada"
)

# Autor principal
df_autor_principal = (
    df_base
    .select(
        "titulo",
        F.trim(F.col("autor")).alias("autor"),
        F.lit("principal").alias("tipo_autoria")
    )
)

# Coautores
df_coautores = (
    df_base
    .filter(
        F.col("coautores").isNotNull() &
        (F.trim(F.col("coautores")) != "")
    )
    .select(
        "titulo",
        F.explode(
            F.split(F.col("coautores"), ",")
        ).alias("autor")
    )
    .withColumn(
        "autor",
        F.trim(F.col("autor"))
    )
    .withColumn(
        "tipo_autoria",
        F.lit("coautor")
    )
)

# Junta todos
df_todos_autores = (
    df_autor_principal
    .unionByName(df_coautores)
)

In [0]:
# Amostra da base de autoria completa, ordenada por título e tipo
# de autoria (inspeção visual).
display(
    df_todos_autores.orderBy("titulo", "tipo_autoria")
)

titulo,autor,tipo_autoria
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Deborah Christina Lopes Costa,coautor
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Nataly Chaves de Freitas,coautor
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Kiara Maia de Oliveira,coautor
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Margaréte May Berkenbrock-Rosito,principal
"""Kit Escola Sem Homofobia"" e Discursos Polí­ticos: quando representações sociais se tornam ideológicas",Moises Romanini,coautor
"""Kit Escola Sem Homofobia"" e Discursos Polí­ticos: quando representações sociais se tornam ideológicas",Ana Carolina Cademartori,coautor
"""Kit Escola Sem Homofobia"" e Discursos Polí­ticos: quando representações sociais se tornam ideológicas",Adriane Roso,principal
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,Emília Rita de Araújo Neta,coautor
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,Erika dos Reis Gusmão Andrade,coautor
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,André Augusto Diniz Lira,principal


In [0]:
# Indicadores da base de autoria: 208 registros de autoria
# (pares artigo × autor), 171 nomes distintos e 89 artigos distintos.
print("Total de registros de autoria:", df_todos_autores.count())

print(
    "Autores distintos:",
    df_todos_autores.select("autor").distinct().count()
)

print(
    "Artigos distintos:",
    df_todos_autores.select("titulo").distinct().count()
)

Total de registros de autoria: 208
Autores distintos: 171
Artigos distintos: 89


In [0]:
# Contagem de participações por autor (ordem decrescente), na grafia
# original. Observação: o valor "-" aparece como "autor" — é um
# placeholder de ausência de coautor, removido adiante.
from pyspark.sql import functions as F

df_autores_contagem = (
    df_todos_autores
    .groupBy("autor")
    .count()
    .orderBy(F.desc("count"), "autor")
)

display(df_autores_contagem)

autor,count
Monica Rabello de Castro,4
Pedro Humberto Faria Campos,4
Rita de Cássia Pereira Lima,4
-,3
Ademir José Rosso,3
Arthur Vianna Ferreira,3
Maria Isabel Antunes-Rocha,3
Romilda Teodora Ens,3
Alcina Maria Testa Braz da Silva,2
Alda Judith Alves-Mazzotti,2


In [0]:
# Cria a coluna "autor_normalizado" (minúsculas, hífens trocados por
# espaço, espaços múltiplos reduzidos) para detectar o mesmo autor
# grafado de formas diferentes.
df_autores_normalizados = (
    df_todos_autores
    .withColumn(
        "autor_normalizado",
        F.lower(
            F.regexp_replace(
                F.regexp_replace(
                    F.trim(F.col("autor")),
                    "-",
                    " "
                ),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
# Detecta autores com mais de uma grafia após a normalização:
# apenas Alda Judith Alves-Mazzotti (com e sem hífen no sobrenome).
display(
    df_autores_normalizados
    .groupBy("autor_normalizado")
    .agg(
        F.collect_set("autor").alias("variacoes"),
        F.countDistinct("autor").alias("qtd_variacoes")
    )
    .filter(F.col("qtd_variacoes") > 1)
)

autor_normalizado,variacoes,qtd_variacoes
alda judith alves mazzotti,"List(Alda Judith Alves-Mazzotti, Alda-Judith Alves-Mazzotti)",2


In [0]:
# Padroniza a grafia detectada na célula anterior: unifica as
# variações de Alda Judith Alves-Mazzotti na forma
# "Alda Judith Alves-Mazzotti".
df_todos_autores_tratado = (
    df_todos_autores
    .withColumn(
        "autor",
        F.when(
            F.lower(
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.col("autor")),
                        "-",
                        " "
                    ),
                    r"\s+",
                    " "
                )
            ) == "alda judith alves mazzotti",
            "Alda Judith Alves-Mazzotti"
        ).otherwise(F.col("autor"))
    )
)

In [0]:
# Resultado da padronização: 170 autores distintos (antes 171 — as
# duas grafias de Alda Judith passam a contar como um único autor).
print(
    "Autores distintos após padronização:",
    df_todos_autores_tratado.select("autor").distinct().count()
)

Autores distintos após padronização: 170


In [0]:
# Primeira gravação da tabela "autores_artigos" — ainda incluindo os
# registros placeholder "-". Sobrescrita pela célula que segue após a
# limpeza (redundância mantida como registro do processo).
df_todos_autores_tratado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.autores_artigos"
    )

In [0]:
# Remove registros de autoria inválidos: nulos, vazios e placeholders
# ("-", "–", "—") que indicam ausência de coautor na planilha.
from pyspark.sql import functions as F

df_todos_autores_tratado = (
    df_todos_autores_tratado
    .filter(
        F.col("autor").isNotNull() &
        (F.trim(F.col("autor")) != "") &
        (~F.trim(F.col("autor")).isin("-", "–", "—"))
    )
)

In [0]:
# Contagem final de participações por autor, já sem os placeholders.
display(
    df_todos_autores_tratado
    .groupBy("autor")
    .count()
    .orderBy(F.desc("count"))
)

autor,count
Monica Rabello de Castro,4
Pedro Humberto Faria Campos,4
Rita de Cássia Pereira Lima,4
Romilda Teodora Ens,3
Maria Isabel Antunes-Rocha,3
Alda Judith Alves-Mazzotti,3
Arthur Vianna Ferreira,3
Ademir José Rosso,3
Viviane Terezinha Koga,2
Andreza Maria de Lima,2


In [0]:
# Gravação final da tabela "tese_bibliometria.silver.autores_artigos"
# (sobrescreve a versão anterior, agora sem os registros placeholder).
df_todos_autores_tratado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.autores_artigos"
    )

## 10. Indicadores de palavras-chave

**Objetivo:** transformar a coluna `palavras_chave` (texto com várias palavras por artigo) em uma base com uma linha por par (artigo, palavra-chave), normalizar grafias/caixa e produzir o indicador de artigos por palavra-chave.

**Dados:** tabela Silver `tese_bibliometria.silver.artigos_bibliometria_tratada` — mesma observação de dependência externa da seção 9.

**Procedimento:** separação das palavras por ponto e vírgula, vÍrgula e travessões (`explode`/`split`); normalização em minúsculas para detecção de variações; padronizações pontuais de grafias identificadas na exploração (ex.: "Teoriadas representações sociais", "covid 19") e unificação de caixa para "Representações Sociais" / "Representação Social".

**Resultado esperado:** base gravada como `tese_bibliometria.silver.palavras_chave_artigos` e indicador final com as palavras presentes em 2+ artigos ("Representações Sociais" lidera com 20 artigos).


In [0]:
# INDICADORES DE PALAVRAS-CHAVE — construção da base artigo × palavra.
# Base: a mesma tabela Silver "artigos_bibliometria_tratada" (dependência
# externa — ver observações finais). A coluna "palavras_chave" é filtrada
# (apenas artigos com valor) e explodida: cada palavra, separada por ponto
# e vírgula, vírgula ou travessões, gera uma linha própria; valores vazios
# após a separação (trim) são descartados.
from pyspark.sql import functions as F

df_base = spark.table(
    "tese_bibliometria.silver.artigos_bibliometria_tratada"
)

df_palavras_chave = (
    df_base
    .filter(
        F.col("palavras_chave").isNotNull() &
        (F.trim(F.col("palavras_chave")) != "")
    )
    .select(
        "titulo",
        F.explode(
            F.split(
                F.col("palavras_chave"),
                r"[.;,–—]+"
            )
        ).alias("palavra_chave")
    )
    .withColumn(
        "palavra_chave",
        F.trim(F.col("palavra_chave"))
    )
    .filter(
        F.col("palavra_chave").isNotNull() &
        (F.col("palavra_chave") != "")
    )
)

In [0]:
# Normalização para agrupamento: minúsculas e espaços múltiplos
# reduzidos (coluna "palavra_chave_normalizada") — detecta a mesma
# palavra grafada com caixa ou espaçamento diferentes.
df_palavras_chave = (
    df_palavras_chave
    .withColumn(
        "palavra_chave_normalizada",
        F.lower(
            F.regexp_replace(
                F.trim(F.col("palavra_chave")),
                r"\s+",
                " "
            )
        )
    )
)

In [0]:
# Amostra da base de palavras-chave por artigo (inspeção visual).
display(
    df_palavras_chave.orderBy("titulo")
)

titulo,palavra_chave
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Experiência Estética
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Narrativa (auto) biográfica
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Representação Social
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",Colcha de Retalhos Digital
"""Colcha de Retalhos Digital"": uma experiência estética das narrativas (auto) biográficas à luz das Representações Sociais",CulturaDigital
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,Formação docente
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,Representações Sociais
A Ciência e os Estudantes de Licenciatura: um estudo comparativo de representações sociais na UFCG,Ciência
A Teoria das Representações Sociais encontra os Estudos Culturais na Indonésia: Buscando novas possibilidades para acomodar estudos emergentes sobre cultura,Localidade
A Teoria das Representações Sociais encontra os Estudos Culturais na Indonésia: Buscando novas possibilidades para acomodar estudos emergentes sobre cultura,Representações Sociais


In [0]:
# Primeiro agrupamento (pela forma normalizada): número de artigos por
# palavra-chave, mantendo as formas originais encontradas — evidencia
# variações como "Representações Sociais" / "Representações sociais".
display(
    df_palavras_chave
    .groupBy("palavra_chave_normalizada")
    .agg(
        F.countDistinct("titulo").alias("quantidade_artigos"),
        F.collect_set("palavra_chave").alias("formas_encontradas")
    )
    .orderBy(F.desc("quantidade_artigos"))
)

palavra_chave_normalizada,quantidade_artigos,formas_encontradas
representações sociais,20,"List(Representações Sociais, Representações sociais)"
representação social,6,"List(Representação Social, Representação social)"
educação,5,List(Educação)
teoria das representações sociais,3,List(Teoria das Representações Sociais)
representaciones sociales,3,"List(Representaciones sociales, Representaciones Sociales)"
metáforas,2,List(Metáforas)
conhecimento,2,List(Conhecimento)
trabajo docente,2,"List(trabajo docente, Trabajo docente)"
docentes,2,List(Docentes)
tecnologia assistiva,2,List(Tecnologia assistiva)


In [0]:
# Renomeia "palavra_chave" para "palavra_chave_original", preservando
# a grafia da planilha antes de criar a versão padronizada.
df_palavras_chave = (
    df_palavras_chave
    .withColumnRenamed(
        "palavra_chave",
        "palavra_chave_original"
    )
)

In [0]:
# Cria a coluna "palavra_chave_padronizada": corrige duas grafias
# problemáticas identificadas na exploração ("Teoriadas representações
# sociais" — texto sem espaços — e "covid 19" → "COVID-19"); as demais
# palavras mantêm a forma original com trim.
from pyspark.sql import functions as F

df_palavras_chave = (
    df_palavras_chave
    .withColumn(
        "palavra_chave_padronizada",
        F.when(
            F.lower(F.trim(F.col("palavra_chave_original")))
            == "teoriadas representações sociais",
            "Teoria das Representações Sociais"
        )
        .when(
            F.lower(F.trim(F.col("palavra_chave_original")))
            == "covid 19",
            "COVID-19"
        )
        .otherwise(
            F.trim(F.col("palavra_chave_original"))
        )
    )
)

In [0]:
# Agrupamento com a padronização parcial: minúsculas e maiúsculas ainda
# separam a mesma palavra (ex.: "Educação" vs. possíveis variantes de caixa).
display(
    df_palavras_chave
    .groupBy("palavra_chave_padronizada")
    .agg(
        F.countDistinct("titulo").alias("quantidade_artigos")
    )
    .orderBy(
        F.desc("quantidade_artigos"),
        "palavra_chave_padronizada"
    )
)

palavra_chave_padronizada,quantidade_artigos
Representações Sociais,15
Educação,5
Representações sociais,5
Teoria das Representações Sociais,4
Representação Social,3
Representação social,3
COVID-19,2
Conhecimento,2
Docentes,2
Formação de Professores,2


In [0]:
# Mesmo agrupamento, filtrando apenas as palavras usadas em 2+ artigos
# (recorte do indicador para leitura).
display(
    df_palavras_chave
    .groupBy("palavra_chave_padronizada")
    .agg(
        F.countDistinct("titulo").alias("quantidade_artigos")
    )
    .filter(F.col("quantidade_artigos") >= 2)
    .orderBy(
        F.desc("quantidade_artigos"),
        "palavra_chave_padronizada"
    )
)

palavra_chave_padronizada,quantidade_artigos
Representações Sociais,15
Educação,5
Representações sociais,5
Teoria das Representações Sociais,4
Representação Social,3
Representação social,3
COVID-19,2
Conhecimento,2
Docentes,2
Formação de Professores,2


In [0]:
# Padronização final de caixa: unifica "Representações sociais" →
# "Representações Sociais" e "Representação social" → "Representação Social",
# consolidando as variações detectadas no agrupamento anterior.
df_palavras_chave = (
    df_palavras_chave
    .withColumn(
        "palavra_chave_padronizada",
        F.when(
            F.lower(F.trim(F.col("palavra_chave_padronizada")))
            == "representações sociais",
            "Representações Sociais"
        )
        .when(
            F.lower(F.trim(F.col("palavra_chave_padronizada")))
            == "representação social",
            "Representação Social"
        )
        .otherwise(
            F.col("palavra_chave_padronizada")
        )
    )
)

In [0]:
# Resultado final do indicador: palavras-chave presentes em 2+ artigos,
# já com caixa unificada ("Representações Sociais" soma 20 artigos).
display(
    df_palavras_chave
    .groupBy("palavra_chave_padronizada")
    .agg(
        F.countDistinct("titulo").alias("quantidade_artigos")
    )
    .filter(F.col("quantidade_artigos") >= 2)
    .orderBy(
        F.desc("quantidade_artigos"),
        "palavra_chave_padronizada"
    )
)

palavra_chave_padronizada,quantidade_artigos
Representações Sociais,20
Representação Social,6
Educação,5
Teoria das Representações Sociais,4
COVID-19,2
Conhecimento,2
Docentes,2
Formação de Professores,2
Metáforas,2
Pandemia,2


In [0]:
# Grava a base de palavras-chave como tabela Silver
# "tese_bibliometria.silver.palavras_chave_artigos" (formato Delta,
# modo overwrite).
df_palavras_chave.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "tese_bibliometria.silver.palavras_chave_artigos"
    )